# Bootstrap 假设检验（CHFS 家庭收入均值检验）

> 对应代码：`MathStatsCode/code_in_notes/Chap.15/bootstrap_htest.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebooks/PyStata.md`。

**实验目的**：对 $H_0:\mu\leq 90000$（我国户均家庭收入不超过 90000 元）这一检验，用 bootstrap 模拟原假设下学生氏化统计量的分布，计算 bootstrap 临界值与 $p$ 值，并与渐近正态近似的结果比较。


首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.15`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致。


In [1]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.15")

import stata_setup
stata_setup.config("/opt/Stata", "mp")


  ___  ____  ____  ____  ____ ®
 /__    /   ____/   /   ____/      Stata 18.0
___/   /   /___/   /   /___/       MP—Parallel Edition

 Statistics and Data Science       Copyright 1985-2023 StataCorp LLC
                                   StataCorp
                                   4905 Lakeway Drive
                                   College Station, Texas 77845 USA
                                   800-782-8272        https://www.stata.com
                                   979-696-4600        service@stata.com

Stata license: Single-user 16-core  perpetual
Serial number: 501806307019
  Licensed to: Keep calm and DO your research
               Jichun Si

Notes:
      1. Unicode is supported; see help unicode_advice.
      2. More than 2 billion observations are allowed; see help obs_advice.
      3. Maximum number of variables is set to 5,000 but can be increased;
          see help set_maxvar.


载入 2017 年 CHFS 家庭数据，删除总收入为负（不适用）的观测，并用 `set seed` 固定随机种子。


In [2]:
%%stata
clear all
use ../datasets/chfs2017_hh.dta
drop if total_income<0
set seed 250211



. clear all

. use ../datasets/chfs2017_hh.dta

. drop if total_income<0
(566 observations deleted)

. set seed 250211

. 


**计算原始样本的t统计量**

先计算原始样本的均值 `theta_hat`（它在“bootstrap 世界”中充当真值）以及检验 $H_0:\mu=90000$ 的 $t$ 统计量 $t=(\bar{x}-90000)/(s/\sqrt{N})$。与正态近似的临界值 $1.65$（单侧）比较，$t\approx1.099$，不能拒绝原假设。


In [3]:
%%stata
su total_income
local theta_hat = r(mean)
local t = (r(mean)-90000)/(r(sd)/sqrt(r(N)))
di "t = " `t'



. su total_income

    Variable |        Obs        Mean    Std. dev.       Min        Max
-------------+---------------------------------------------------------
total_income |     39,445    91065.53    192618.2          0    5000000

. local theta_hat = r(mean)

. local t = (r(mean)-90000)/(r(sd)/sqrt(r(N)))

. di "t = " `t'
t = 1.0986591

. 


**bootstrap学生氏化统计量并保存结果**

`bootstrap (...), reps(1000) saving(bootstrap_htest.dta, replace): su total_income`：每次有放回重抽样后重新计算均值与标准差，并收集学生氏化统计量 $t^{*}=\sqrt{N}(\bar{x}^{*}-\bar{x})/s^{*}$，1000 次重复的结果保存到 `bootstrap_htest.dta` 备用。

注意这里 $t^{*}$ 的中心是原始样本均值 $\bar{x}$ 而不是 90000：在“bootstrap 世界”中真值是 $\bar{x}$，这样 $t^{*}$ 所对应的“原假设”在 bootstrap 世界中恰好成立，从而 $\{t_b^{*}\}$ 的分布可以用来近似原假设下 $t$ 的分布。


In [4]:
%%stata
bootstrap (sqrt(r(N))*(r(mean)-`theta_hat')/r(sd)), reps(1000) saving(bootstrap_htest.dta, replace): su total_income


(running summarize on estimation sample)

         excluded from the resampling because of missing values or other
         reasons. To exclude observations, press Break, save the data, drop
         any observations that are to be excluded, and rerun bootstrap.


(file bootstrap_htest.dta not found)

Bootstrap replications (1,000): 

.

.

..

...

.

.

1,000 done

Bootstrap results                                       Number of obs = 39,445
                                                        Replications  =  1,000

      Command: summarize total_income
        _bs_1: sqrt(r(N))*(r(mean)-91065.52641821354)/r(sd)



------------------------------------------------------------------------------


             |   Observed   Bootstrap                         Normal-based
             | coefficient  std. err.      z    P>|z|     [95% conf. interval]
-------------+----------------------------------------------------------------
       _bs_1 |          0   1.023505     0.00   1.000    -2.006034    2.006034
------------------------------------------------------------------------------


**载入bootstrap结果**

`preserve` 暂存当前数据，`use bootstrap_htest.dta` 载入 1000 个 $t^{*}$，并把自动命名的 `_bs_1` 改名为 `tstar`。


In [5]:
%%stata
preserve
use bootstrap_htest.dta, clear
rename _bs_1 tstar



. preserve

. use bootstrap_htest.dta, clear
(bootstrap: summarize)

. rename _bs_1 tstar



. 


**bootstrap临界值**

`_pctile tstar, p(2.5 95 97.5)` 计算 $\{t_b^{*}\}$ 的三个分位数：95% 分位数即 5% 显著性水平下右侧检验的 bootstrap 临界值；2.5% 与 97.5% 分位数即双侧检验的 bootstrap 临界值。与渐近正态的 $1.65$ 和 $\pm1.96$ 对比，可以看到 bootstrap 分布略微左偏（左侧尾部更厚），这反映了右偏收入分布下样本均值抽样分布的左偏性。


In [6]:
%%stata
_pctile tstar, p(2.5 95 97.5)
di "右侧检验临界值 = " r(r2)
di "双侧检验临界值 = [" r(r1) ", " r(r3) "]"



. _pctile tstar, p(2.5 95 97.5)

. di "右侧检验临界值 = " r(r2)
右侧检验临界值 = 1.6767543

. di "双侧检验临界值 = [" r(r1) ", " r(r3) "]"
双侧检验临界值 = [-2.026365, 1.9254783]

. 


**bootstrap p值**

bootstrap $p$ 值即 $\{t_b^{*}\}$ 中比实际统计量更极端的比例：右侧检验为 $t^{*}>t$ 的比例；双侧检验使用对称 $p$ 值，即 $|t^{*}|>|t|$ 的比例。可以看到，两个 $p$ 值都比渐近正态近似的结果（0.136 和 0.2719）略大。


In [7]:
%%stata
count if tstar>`t'
di "右侧检验p值 = " r(N)/_N
count if abs(tstar)>abs(`t')
di "双侧检验p值 = " r(N)/_N



. count if tstar>`t'
  161

. di "右侧检验p值 = " r(N)/_N
右侧检验p值 = .161

. count if abs(tstar)>abs(`t')
  301

. di "双侧检验p值 = " r(N)/_N
双侧检验p值 = .301

. 


最后 `restore` 恢复原始数据，并删除中间文件 `bootstrap_htest.dta`，保持工作目录整洁。


In [8]:
%%stata
restore
rm bootstrap_htest.dta



. restore

. rm bootstrap_htest.dta

. 
